In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "2. Silver Schema")
dbutils.widgets.text("gold_schema", "valeriimatviiv_gold", "3. Gold Schema")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

In [0]:
# 1. Compact & Z-Order Silver Price Table
# print("Optimizing Silver Price table...")
spark.sql(f"""
    OPTIMIZE {catalog}.{silver_schema}.nasdaq_price_silver 
    ZORDER BY (Symbol, TradeTimestamp)
""")

# 2. Compact & Z-Order Silver News Table
# print("Optimizing Silver News table...")
spark.sql(f"""
    OPTIMIZE {catalog}.{silver_schema}.finnhub_news_silver 
    ZORDER BY (Symbol, NewsTimestamp)
""")

# 3. Compact & Z-Order Gold Impact Table
# print("Optimizing Gold Impact table...")
spark.sql(f"""
    OPTIMIZE {catalog}.{gold_schema}.nasdaq_news_impact_gold 
    ZORDER BY (Symbol, NewsTimestamp)
""")

# 4. Vacuum Stale Delta Commit Files (Retain 7 Days / 168 Hours)
# print("Running Vacuum cleanup...")
spark.sql(f"VACUUM {catalog}.{silver_schema}.nasdaq_price_silver RETAIN 168 HOURS")
spark.sql(f"VACUUM {catalog}.{silver_schema}.finnhub_news_silver RETAIN 168 HOURS")
spark.sql(f"VACUUM {catalog}.{gold_schema}.nasdaq_news_impact_gold RETAIN 168 HOURS")

# print("Maintenance job completed successfully.")

In [0]:
# # Verify Delta Table Commit History for OPTIMIZE and VACUUM actions
# history_df = spark.sql(f"DESCRIBE HISTORY {catalog}.{gold_schema}.nasdaq_news_impact_gold")
# display(history_df.select("version", "timestamp", "operation", "operationParameters").limit(3))